# ISL Coreference Dataset: Pair Construction

This notebook takes the raw coreference annotations produced by `build_coref_dataset.py`
and constructs the final (anchor, candidate) pair dataset used for model training and evaluation.

**Pipeline overview:**
1. Filter high-confidence coreference rows and pull their antecedents
2. Attach video folder paths (from iSign 2.0 metadata)
3. Build (uid_A, uid_B) pairs with binary coreference labels
4. Undersample negatives (k negatives per positive)
5. Split by video ID into train / val / test



In [ ]:
import re
import numpy as np
import pandas as pd

## 1. Load annotations and filter by confidence

In [ ]:
COREF_PATH   = "data/coref.csv"
ISIGN_PATH   = "data/english_sentences_with_folders.csv"
TRAIN_PATH   = "datasets/train.csv"
VAL_PATH     = "datasets/val.csv"
TEST_PATH    = "datasets/test.csv"
CONFIDENCE_THRESHOLD = 0.95
NEG_PER_POS = 6
RANDOM_SEED = 42

In [ ]:
df = pd.read_csv(COREF_PATH)
print(f"Loaded {len(df):,} rows")
df.head()

In [ ]:
# Keep only high-confidence coreference rows
high_conf = df[df["confidence"] > CONFIDENCE_THRESHOLD]
print(f"High-confidence rows (>{CONFIDENCE_THRESHOLD}): {len(high_conf):,}")

# Pull the antecedent clips those rows point to
antecedent_uids = high_conf["resolves_to_uid"].dropna().unique()
antecedent_rows = df[df["uid"].isin(antecedent_uids)]
print(f"Antecedent clips retrieved: {len(antecedent_rows):,}")

ds = (
    pd.concat([high_conf, antecedent_rows])
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"Combined (deduplicated): {len(ds):,} rows")

## 2. Sort by video and clip order

In [ ]:
def clip_sort_key(uid):
    """Extract numeric clip index from uid strings like 'abc_clip_3' or 'abc--3'."""
    m = re.search(r"_clip_(\d+)$", uid)
    if m:
        return int(m.group(1))
    m = re.search(r"--(\-?\d+)$", uid)
    if m:
        return int(m.group(1))
    return 0

ds["_sort_key"] = ds["uid"].apply(clip_sort_key)
ds = ds.sort_values(["video_id", "_sort_key"]).drop(columns="_sort_key").reset_index(drop=True)
ds.head()

## 3. Attach video folder paths from iSign 2.0 metadata

In [ ]:
isign = pd.read_csv(ISIGN_PATH)

# one folder per youtube id
id_to_folder = (
    isign[["you_tube_id", "video_folder"]]
    .drop_duplicates(subset=["you_tube_id"])
    .set_index("you_tube_id")["video_folder"]
)

ds["video_folder"] = ds["video_id"].map(id_to_folder)

n_unmatched = ds["video_folder"].isna().sum()
print(f"Unmatched video IDs: {n_unmatched}")
if n_unmatched:
    print(ds[ds["video_folder"].isna()][["video_id"]].drop_duplicates())

In [ ]:
# Drop annotation meta-columns no longer needed
ds = ds.drop(columns=["has_coreference", "needs_review"])

# Drop trivial cases where mention == resolved entity (exact string match)
before = len(ds)
ds = ds[ds["mention"] != ds["resolved_entity"]].reset_index(drop=True)
print(f"Dropped {before - len(ds)} rows where mention == resolved_entity")



## 4. Build (anchor, candidate) pairs

In [ ]:
pairs = []

for video_id, group in ds.groupby("video_id"):
    # antecedent rows have no confidence (they were pulled in as anchors)
    antecedents = group[group["confidence"].isna()]

    for _, ant in antecedents.iterrows():
        for _, row in group.iterrows():
            if row["uid"] == ant["uid"]:
                continue

            is_positive = row["resolves_to_uid"] == ant["uid"]

            pairs.append({
                "video_id":        video_id,
                "uid_A":           ant["uid"],
                "text_A":          ant["text"],
                "uid_B":           row["uid"],
                "text_B":          row["text"],
                "mention":         row["mention"]         if is_positive else None,
                "resolved_entity": row["resolved_entity"] if is_positive else None,
                "confidence":      row["confidence"]      if is_positive else None,
                "has_coreference": int(is_positive),
            })

pairs_df = pd.DataFrame(pairs)
print(pairs_df["has_coreference"].value_counts())

In [ ]:
# Drop anchor UIDs that have no positive match at all
valid_anchors  = pairs_df[pairs_df["has_coreference"] == 1]["uid_A"].unique()
pairs_df_clean = pairs_df[pairs_df["uid_A"].isin(valid_anchors)].reset_index(drop=True)

print(f"Before anchor filter: {len(pairs_df):,}")
print(f"After:                {len(pairs_df_clean):,}")
print(pairs_df_clean["has_coreference"].value_counts())

## 5. Undersample negatives

In [ ]:
pos = pairs_df_clean[pairs_df_clean["has_coreference"] == 1]
neg = pairs_df_clean[pairs_df_clean["has_coreference"] == 0]

neg_sampled = (
    neg.groupby("uid_A")
    .apply(lambda x: x.sample(min(NEG_PER_POS, len(x)), random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

pairs_final = pd.concat([pos, neg_sampled]).reset_index(drop=True)
print(pairs_final["has_coreference"].value_counts())



## 6. Train / val / test split (by video ID)

In [ ]:
import os
os.makedirs("datasets", exist_ok=True)

np.random.seed(RANDOM_SEED)

video_ids = pairs_final["video_id"].unique()
np.random.shuffle(video_ids)

n_test = max(1, int(len(video_ids) * 0.10))
n_val  = max(1, int(len(video_ids) * 0.10))

test_vids  = set(video_ids[:n_test])
val_vids   = set(video_ids[n_test : n_test + n_val])
train_vids = set(video_ids[n_test + n_val:])

train_df = pairs_final[pairs_final["video_id"].isin(train_vids)].reset_index(drop=True)
val_df   = pairs_final[pairs_final["video_id"].isin(val_vids)].reset_index(drop=True)
test_df  = pairs_final[pairs_final["video_id"].isin(test_vids)].reset_index(drop=True)

print(f"Train: {len(train_df):,} pairs  ({len(train_vids)} videos)")
print(f"Val:   {len(val_df):,} pairs  ({len(val_vids)} videos)")
print(f"Test:  {len(test_df):,} pairs  ({len(test_vids)} videos)")

train_df.to_csv(TRAIN_PATH, index=False)
val_df.to_csv(VAL_PATH,   index=False)
test_df.to_csv(TEST_PATH,  index=False)

print("Saved train / val / test splits.")